# Master Notebook for Edge Classification (Branch Congestion)
This master notebook trains and compares multiple Graph Neural Network architectures (GCN, GraphSAGE, GAT, MPNN, ECGNN, PINN) to predict congestion across the branches of the IEEE 57-bus grid.

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "pandas>=2.2.3", "pandapower", "numpy", "torch", "torchvision",
    "torchaudio", "torch_geometric", "h5py", "optuna", "matplotlib"])

0

In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split, WeightedRandomSampler
import pandapower.networks as nw

c:\Users\i34005\OneDrive - Wood Mackenzie Limited\CSML\csml-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. PyTorch Geometric Dataset

In [3]:
from ieee_dataset import IEEECongestionCSVDataset

csv_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "congestion_dataset_v5.csv")
dataset = IEEECongestionCSVDataset(csv_path)

train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size]
)

# --- WeightedRandomSampler: oversample congested scenarios ---
cong_cols = dataset.df.iloc[:, 171:]  # 80 congestion target columns
has_congestion = (cong_cols.sum(axis=1) > 0).values  # True if any branch congested

train_indices = train_dataset.indices
train_labels = has_congestion[train_indices]
n_pos = train_labels.sum()
n_neg = len(train_labels) - n_pos
weight_pos = len(train_labels) / (2 * n_pos + 1e-8)
weight_neg = len(train_labels) / (2 * n_neg + 1e-8)
sample_weights = torch.tensor(
    [weight_pos if lbl else weight_neg for lbl in train_labels],
    dtype=torch.float64
)
train_sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_dataset), replacement=True)
print(f"Congested samples in train: {n_pos}/{len(train_labels)} ({100*n_pos/len(train_labels):.2f}%)")
print(f"Sample weights — pos: {weight_pos:.1f}, neg: {weight_neg:.1f}")

train_loader = DataLoader(train_dataset, batch_size=256, sampler=train_sampler)
val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dataset: {len(dataset)} samples  (train={train_size}, val={val_size}, test={test_size})")
print(f"Using device: {device}")

Congested samples in train: 2732/35000 (7.81%)
Sample weights — pos: 6.4, neg: 0.5
Dataset: 50000 samples  (train=35000, val=7500, test=7500)
Using device: cpu


2. Hyperparameter Tuning (Optuna)

Quick search (few epochs per trial) to find optimal hyperparameters for each model.

In [ ]:
from models.gcn_model import GCNEdgePredictor
from models.graphsage_model import GraphSAGEEdgePredictor
from models.gat_model import GATEdgePredictor
from models.mpnn_model import MPNNEdgePredictor
from models.ecgnn_model import ECGNNEdgePredictor
from models.pinn_model import PINNEdgePredictor
import pickle
import optuna

MODEL_TYPES = ['GCN', 'GraphSAGE', 'GAT', 'MPNN', 'ECGNN', 'PINN']
MODEL_TYPES = ['GraphSAGE', 'GAT', 'MPNN', 'ECGNN', 'PINN']

model_classes = {
    'GCN': GCNEdgePredictor,
    'GraphSAGE': GraphSAGEEdgePredictor,
    'GAT': GATEdgePredictor,
    'MPNN': MPNNEdgePredictor,
    'ECGNN': ECGNNEdgePredictor,
    'PINN': PINNEdgePredictor
}

IN_CHANNELS = 5       # Pd, Qd, gen_status, pgmax, pgmin
NUM_EDGE_FEATS = 3    # smax, branch_status, susceptance (b)

OPTUNA_EPOCHS = 10
OPTUNA_TRIALS = 1  # Increase for a more thorough search

best_hyperparams = {}

for TUNE_MODEL in MODEL_TYPES:
    print(f"\n{'='*50}")
    print(f"Tuning hyperparameters for {TUNE_MODEL}...")
    print(f"{'='*50}")

    OptunaModelClass = model_classes[TUNE_MODEL]

    def objective(trial):
        hidden_channels = trial.suggest_categorical("hidden_channels", [32, 64, 128])
        batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.4)
        pw_value = trial.suggest_float("pos_weight", 1.0, 50.0, log=True)

        tune_train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler)
        tune_val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        trial_model = OptunaModelClass(
            in_channels=IN_CHANNELS,
            hidden_channels=hidden_channels,
            branch_u=dataset.branch_u,
            branch_v=dataset.branch_v,
            dropout_rate=dropout_rate,
            num_edge_feats=NUM_EDGE_FEATS,
        ).to(device)

        opt = torch.optim.Adam(trial_model.parameters(), lr=lr)
        pos_w = torch.tensor([pw_value]).to(device)
        crit = nn.BCEWithLogitsLoss(reduction='none', pos_weight=pos_w)

        for epoch in range(OPTUNA_EPOCHS):
            trial_model.train()
            for batch in tune_train_loader:
                batch = batch.to(device)
                opt.zero_grad()
                out = trial_model(batch)
                y = batch.y.view(-1, 1)
                mask = batch.status.view(-1, 1)
                loss = (crit(out, y) * mask).sum() / (mask.sum() + 1e-8)
                loss.backward()
                opt.step()

        trial_model.eval()
        tp, fp, fn = 0, 0, 0
        with torch.no_grad():
            for batch in tune_val_loader:
                batch = batch.to(device)
                out = trial_model(batch)
                y = batch.y.view(-1, 1)
                mask = batch.status.view(-1, 1)
                preds = (out > 0).float()
                masked_preds = preds * mask
                masked_y = y * mask
                tp += ((masked_preds == 1) & (masked_y == 1)).float().sum().item()
                fp += ((masked_preds == 1) & (masked_y == 0)).float().sum().item()
                fn += ((masked_preds == 0) & (masked_y == 1)).float().sum().item()
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        return f1

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=OPTUNA_TRIALS)
    print(f"\nBest val F1 for {TUNE_MODEL}: {study.best_value:.4f}")
    print(f"Best params for {TUNE_MODEL}: {study.best_params}")

    best_hyperparams[TUNE_MODEL] = {
        'best_val_f1': study.best_value,
        'params': study.best_params
    }

print("\n\n--- OPTUNA RESULTS SUMMARY ---")
for model_name, info in best_hyperparams.items():
    print(f"{model_name} (F1: {info['best_val_f1']:.4f})  Params: {info['params']}")


Tuning hyperparameters for GraphSAGE...


[I 2026-04-09 16:11:44,423] A new study created in memory with name: no-name-ebff7e16-e929-45f4-a38b-52d55b57c838
[I 2026-04-09 16:19:25,181] Trial 0 finished with value: 0.45348398895291203 and parameters: {'hidden_channels': 32, 'batch_size': 256, 'lr': 0.00204244211716237, 'dropout_rate': 0.06224272772046602, 'pos_weight': 44.5756846562434}. Best is trial 0 with value: 0.45348398895291203.
[I 2026-04-09 16:32:57,627] Trial 1 finished with value: 0.30677702788797856 and parameters: {'hidden_channels': 128, 'batch_size': 512, 'lr': 0.0002845009906255652, 'dropout_rate': 0.1582992312926006, 'pos_weight': 16.677312335744507}. Best is trial 0 with value: 0.45348398895291203.
[I 2026-04-09 16:44:47,104] Trial 2 finished with value: 0.7233782083084891 and parameters: {'hidden_channels': 64, 'batch_size': 128, 'lr': 0.0032564455793903033, 'dropout_rate': 0.00831861317375413, 'pos_weight': 2.9003142010354384}. Best is trial 2 with value: 0.7233782083084891.
[I 2026-04-09 16:58:22,815] Trial 

KeyboardInterrupt: 

3. Full Training with Optimal Hyperparameters

Train each model for 100 epochs using the best hyperparameters found by Optuna, then evaluate on the test set and save to `models/trained/`.

In [ ]:
TRAINED_DIR = os.path.join("models", "trained")
os.makedirs(TRAINED_DIR, exist_ok=True)

epochs = 100
all_histories = {}
all_test_metrics = {}

for m_type in MODEL_TYPES:
    params = best_hyperparams[m_type]['params']
    print(f"\n{'='*40}\nTraining {m_type} with optimal params\n{'='*40}")
    print(f"  hidden={params['hidden_channels']}, lr={params['lr']:.5f}, "
          f"dropout={params['dropout_rate']:.3f}, pos_weight={params['pos_weight']:.2f}, "
          f"batch_size={params['batch_size']}")

    ModelClass = model_classes[m_type]

    model = ModelClass(
        in_channels=IN_CHANNELS,
        hidden_channels=params['hidden_channels'],
        branch_u=dataset.branch_u,
        branch_v=dataset.branch_v,
        dropout_rate=params['dropout_rate'],
        num_edge_feats=NUM_EDGE_FEATS,
    ).to(device)

    train_ldr = DataLoader(train_dataset, batch_size=params['batch_size'], sampler=train_sampler)
    val_ldr   = DataLoader(val_dataset,   batch_size=params['batch_size'], shuffle=False)
    test_ldr  = DataLoader(test_dataset,  batch_size=params['batch_size'], shuffle=False)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'])
    pos_weight = torch.tensor([params['pos_weight']]).to(device)
    criterion = nn.BCEWithLogitsLoss(reduction='none', pos_weight=pos_weight)

    history = {"epoch": [], "train_loss": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        total_loss, n_batches = 0.0, 0
        for batch in train_ldr:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch)
            y = batch.y.view(-1, 1)
            mask = batch.status.view(-1, 1)
            loss = (criterion(out, y) * mask).sum() / (mask.sum() + 1e-8)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        avg_loss = total_loss / n_batches

        # --- Validate ---
        model.eval()
        correct, total_active = 0, 0
        v_tp, v_fp, v_fn = 0, 0, 0
        with torch.no_grad():
            for batch in val_ldr:
                batch = batch.to(device)
                out = model(batch)
                y = batch.y.view(-1, 1)
                mask = batch.status.view(-1, 1)
                preds = (out > 0).float()
                masked_preds = preds * mask
                masked_y = y * mask
                correct += ((preds == y).float() * mask).sum().item()
                total_active += mask.sum().item()
                v_tp += ((masked_preds == 1) & (masked_y == 1)).float().sum().item()
                v_fp += ((masked_preds == 1) & (masked_y == 0)).float().sum().item()
                v_fn += ((masked_preds == 0) & (masked_y == 1)).float().sum().item()
        val_acc = correct / total_active
        val_prec = v_tp / (v_tp + v_fp + 1e-8) if v_tp + v_fp > 0 else 0
        val_rec = v_tp / (v_tp + v_fn + 1e-8) if v_tp + v_fn > 0 else 0

        history["epoch"].append(epoch)
        history["train_loss"].append(avg_loss)
        history["val_acc"].append(val_acc)

        if epoch % 10 == 0 or epoch == epochs:
            print(f"Epoch {epoch:3d}/{epochs} | loss={avg_loss:.4f} | val_acc={val_acc:.4f} | precision={val_prec:.4f} | recall={val_rec:.4f}")

    all_histories[m_type] = history

    # --- Test ---
    model.eval()
    correct, total_active = 0, 0
    tp, fp, fn = 0, 0, 0
    with torch.no_grad():
        for batch in test_ldr:
            batch = batch.to(device)
            out = model(batch)
            y = batch.y.view(-1, 1)
            mask = batch.status.view(-1, 1)
            preds = (out > 0).float()
            masked_preds = preds * mask
            masked_y = y * mask
            correct += ((preds == y).float() * mask).sum().item()
            total_active += mask.sum().item()
            tp += ((masked_preds == 1) & (masked_y == 1)).float().sum().item()
            fp += ((masked_preds == 1) & (masked_y == 0)).float().sum().item()
            fn += ((masked_preds == 0) & (masked_y == 1)).float().sum().item()

    test_acc = correct / total_active if total_active > 0 else 0
    precision = tp / (tp + fp + 1e-8) if tp + fp > 0 else 0
    recall = tp / (tp + fn + 1e-8) if tp + fn > 0 else 0
    f1 = 2 * precision * recall / (precision + recall + 1e-8) if precision + recall > 0 else 0

    all_test_metrics[m_type] = {
        "acc": test_acc, "precision": precision, "recall": recall, "f1": f1
    }
    print(f"Test Accuracy: {test_acc:.4f} | F1: {f1:.4f}")

    # --- Save model ---
    save_path = os.path.join(TRAINED_DIR, f"{m_type}.pkl")
    with open(save_path, "wb") as f:
        pickle.dump(model.cpu().state_dict(), f)
    model.to(device)
    print(f"Model saved to {save_path}")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for m_type, hist in all_histories.items():
    ax1.plot(hist["epoch"], hist["train_loss"], label=m_type)
    ax2.plot(hist["epoch"], hist["val_acc"], label=m_type)

ax1.set(xlabel="Epoch", ylabel="Loss", title="Training Loss")
ax1.legend()
ax2.set(xlabel="Epoch", ylabel="Accuracy", title="Validation Accuracy")
ax2.legend()
fig.tight_layout()
plt.show()

print("\n--- Test Set Results (trained with Optuna-tuned hyperparameters) ---")
metrics_df = pd.DataFrame(all_test_metrics).T
metrics_df = metrics_df[['acc', 'f1', 'precision', 'recall']]
display(metrics_df.sort_values(by='f1', ascending=False))